# Penjelasan lokal ala LIME
**tabular › explain_classification** · notebook `local_surrogate`

**Kapan dipakai:** Menjelaskan satu prediksi dengan model linear sederhana di sekitar baris itu. Tanpa library tambahan.

**Jebakan umum:** Versi sederhana (bukan paket LIME asli): perturbasi Gaussian pada ruang fitur terproses. Hasil bergantung lebar kernel dan seed; jalankan beberapa seed dan lihat kestabilan.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/classification/titanic_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/classification/titanic_test.csv"      # None kalau tidak ada file test
TARGET     = "survived"
ID_COL     = 'id'
DROP_COLS  = ['alive']
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "local_surrogate"
TASK       = "classification"
KIND       = "tree"
OUT = out_dir("tabular", "explain_classification", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("kelas:", dict(enumerate(le.classes_)))
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model yang akan dijelaskan
Penjelasan selalu dihitung pada **validation**, bukan training.

In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
pipe = build_pipeline(model, NUM, CAT, kind='tree').fit(Xtr, ytr)

In [ ]:
pred = pipe.predict(Xva)
proba = pipe.predict_proba(Xva)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(proba.shape[1]))
display(show_table(metrics))

Matriks fitur setelah preprocessing (nama kolom dipertahankan). `mdl` = model di dalam pipeline.

In [ ]:
from superdb.tabular import feature_names
names = feature_names(pipe)
pre, mdl = pipe.named_steps["pre"], pipe.named_steps["model"]
Ztr = pd.DataFrame(pre.transform(Xtr), columns=names, index=Xtr.index)
Zva = pd.DataFrame(pre.transform(Xva), columns=names, index=Xva.index)
Zva.shape

## 5. Perturbasi di sekitar satu baris
Kita ambil baris `i`, bangkitkan banyak titik acak di dekatnya, minta model memprediksi, lalu cocokkan model linear berbobot jarak.

In [ ]:
i = 0                                                    # baris yang dijelaskan (ubah sesuai kebutuhan)
x0 = Zva.iloc[i].to_numpy(float); sd = Ztr.std().replace(0, 1).to_numpy(float)
rng = np.random.default_rng(SEED); N = 3000
Zp = x0 + rng.normal(size=(N, len(x0))) * sd * 0.5      # 0.5 = lebar lingkungan (coba 0.25 / 1.0)
f = (lambda Z: mdl.predict_proba(pd.DataFrame(Z, columns=Ztr.columns))[:, 1]) if TASK == "classification" else (lambda Z: mdl.predict(pd.DataFrame(Z, columns=Ztr.columns)))
yp = f(Zp)
d2 = (((Zp - x0) / sd) ** 2).sum(1); w = np.exp(-d2 / (2 * (0.75 * np.sqrt(len(x0))) ** 2))
from sklearn.linear_model import Ridge
lin = Ridge(alpha=1.0).fit((Zp - x0) / sd, yp, sample_weight=w)
print(f"prediksi model untuk baris ini: {f(x0.reshape(1, -1))[0]:.4f}   |   R² lokal penjelas: {lin.score((Zp - x0) / sd, yp, sample_weight=w):.3f}")
coef = pd.Series(lin.coef_, index=Ztr.columns).sort_values(key=abs, ascending=False).head(10)[::-1]

## 6. Fitur yang paling memengaruhi prediksi baris ini
Koefisien positif = menaikkan prediksi di lingkungan baris ini. Nilai fitur baris ada di tabel bawah.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4)); coef.plot.barh(ax=ax, color=["tab:red" if c > 0 else "tab:blue" for c in coef])
ax.set_title("Penjelasan lokal (surrogate linear)"); plt.tight_layout(); plt.savefig(OUT / "local_explain.png", dpi=130); plt.show()
display(Zva.iloc[[i]][coef.index[::-1]].T.rename(columns={Zva.index[i]: "nilai baris"}).round(3))

In [ ]:
save_json({"valid": metrics, "method": "local_surrogate"}, OUT / "metrics.json")
save_model({"pipe": pipe, "cols": cols}, OUT / "model.joblib"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Penjelasan model adalah bahan bab *analisis* di laporan: tulis 3 temuan utama + satu contoh kasus.
- Fitur yang 'terlalu penting' (mis. satu kolom mendominasi) = curigai kebocoran (lihat `notebooks/tabular/validation/`).